In [0]:
%sql
SET school.dataset_path='abfss://unity-catalog-storage@dbstorageqh3nvwxzzfwa6.dfs.core.windows.net/7405605315193329/mnt/DE-Associate-Book/datasets/school/'

Path to source dataset is now set.
<p>Let's now create a bronze table to ingest the CDC feed. Perfect use case for Auto Loader.</p>

In [0]:
%sql
CREATE OR REFRESH STREAMING TABLE courses_bronze
COMMENT "The raw courses data, ingested from CDC feed."
AS SELECT * FROM cloud_files("${school.dataset_path}/courses-cdc", "json")

Now silver time, apply changes from CDC feed to maintain up-to-date course data.
<p>We need to create the table before applying changes. APPLY INTO requires a table to be created.</p>

In [0]:
CREATE OR REFRESH STREAMING TABLE courses_silver;

Once the silver table is declared, use APPLY CHANGES INTO command to specify how changes from bronze table should be applied.

In [0]:
APPLY CHANGES INTO LIVE.courses_silver
FROM STREAM(LIVE.courses_bronze)
KEYS (course_id)
APPLY AS DELETE WHEN row_status = "DELETE"
SEQUENCE BY row_time
COLUMNS * EXCEPT (row_status, row_time)

We've now got our bronze and silver, let's aggregate course count by teacher for gold.

In [0]:
CREATE OR REPLACE MATERIALIZED VIEW instruct_counts_stats
COMMENT "Number of courses per instructor"
AS SELECT instructor, count(*) as courses_count,
          current_timestamp() as updated_time
FROM LIVE.courses_silver
GROUP BY instructor

In [0]:
CREATE TEMPORARY LIVE VIEW courses_sales
AS 
SELECT b.title, o.quantity
FROM (
  SELECT *, explode(courses) AS course
  FROM LIVE.enrollments_cleaned
) o
INNER JOIN LIVE.courses_silver b
ON o.course.course_id = b.course_id;